In [0]:
# COMMAND ----------

# MAGIC %md
# MAGIC # 3. Feature Engineering
# MAGIC Prepare features with validated data only

# COMMAND ----------

%pip install polars pandas scikit-learn

# COMMAND ----------

import polars as pl
import pandas as pd
import numpy as np

print(" Libraries loaded")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Load & Validate Data

# COMMAND ----------

judges = pl.read_csv("/Workspace/Users/sumedha190198@gmail.com/judicial-backlog/judges_clean.csv")
disp_key = pl.read_csv("/Workspace/Users/sumedha190198@gmail.com/judicial-backlog/disp_name_key.csv")
type_key = pl.read_csv("/Workspace/Users/sumedha190198@gmail.com/judicial-backlog/type_name_key.csv")
purpose_key = pl.read_csv("/Workspace/Users/sumedha190198@gmail.com/judicial-backlog/purpose_name_key.csv")

lf_sample = pl.scan_parquet("/Workspace/Users/sumedha190198@gmail.com/judicial-backlog/cases_sample_10pct.parquet")

judges = judges.with_columns([
    pl.col("start_date").str.to_date("%d-%m-%Y", strict=False).alias("judge_start"),
    pl.col("end_date").str.to_date("%d-%m-%Y", strict=False).alias("judge_end"),
])

judges_valid = judges.filter(pl.col("judge_start").is_not_null())
judges_lazy = judges_valid.lazy()

lf_filtered = (
    lf_sample
    .filter((pl.col("duration_days") >= 0) & (pl.col("duration_days") <= 7300))
    .join(judges_lazy.select(["state_code", "dist_code", "court_no", "judge_position", "judge_start"]),
          on=["state_code", "dist_code", "court_no", "judge_position"], how="left")
    .with_columns([(pl.col("filing_date").dt.year() - pl.col("judge_start").dt.year()).alias("judge_tenure_years")])
    .filter(pl.col("judge_start").is_not_null())
    .filter(pl.col("filing_date") >= pl.col("judge_start"))
    .filter(pl.col("judge_tenure_years") >= 0)
    .filter(pl.col("judge_tenure_years") <= 50)
)

print(" Data loaded with validation")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Build Feature Pipeline

# COMMAND ----------

# Cast types and prepare for ML
lf_features = (
    lf_filtered
    .with_columns([
        pl.col("disp_name").cast(pl.Int64),
        pl.col("type_name").cast(pl.Int64),
        pl.col("purpose_name").cast(pl.Int64),
    ])
    .join(judges_lazy.select(["state_code", "dist_code", "court_no", "judge_position", "judge_start"]),
          on=["state_code", "dist_code", "court_no", "judge_position"], how="left")
    .with_columns([(pl.col("filing_date").dt.year() - pl.col("judge_start").dt.year()).alias("judge_tenure_years")])
    # VALIDATION: Ensure only valid tenure
    .filter(pl.col("judge_tenure_years") >= 0)
    .filter(pl.col("judge_tenure_years") <= 50)
)

print(f" Features engineered with validation")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Features Created

# COMMAND ----------

print(f"""
FEATURES ENGINEERED (VALIDATED)
===============================
 judge_tenure_years: Judge experience (0-50 years)
 disp_name (int64): Case disposition code
 type_name (int64): Case type code
 purpose_name (int64): Case purpose code
 female_defendant: Defendant gender
 female_petitioner: Petitioner gender
 duration_days: Case duration (regression target)
 long_running: Long-running flag (classification target)
""")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
 Libraries loaded
 Data loaded with validation
 Features engineered with validation

FEATURES ENGINEERED (VALIDATED)
 judge_tenure_years: Judge experience (0-50 years)
 disp_name (int64): Case disposition code
 type_name (int64): Case type code
 purpose_name (int64): Case purpose code
 female_defendant: Defendant gender
 female_petitioner: Petitioner gender
 duration_days: Case duration (regression target)
 long_running: Long-running flag (classification target)

